# prefit on Telco Customer Churn — what it finds, what it misses

A validation notebook for prefit 0.4, written after the Red Wine Quality guide
(`wine_quality_guide.ipynb`, prefit 0.3). It asks three questions, in this order:

1. **Which findings change a decision?** Public Kaggle EDAs of this dataset report a handful of findings that
   change what you do (how to read a column, what to drop, how to split, which metric). We list them first,
   *before* looking at prefit's output, and then check each one against what prefit reports: **caught**,
   **missed**, or **listed without changing anything** (noise).
2. **Missing values.** `TotalCharges` has 11 blanks. Are they unknown, or something else? This is the case
   that the 0.4 missing-data analysis was built for.
3. **The flow.** How many lines does it take to go from the raw frame to an honest test score?

Everything below is computed in this run; the comparison verdicts are computed from the report, not typed in.

In [1]:
import warnings

import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import HistGradientBoostingClassifier

import prefit as pf

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 220)
print(f"prefit {pf.__version__} · scikit-learn {sklearn.__version__} · pandas {pd.__version__}")

prefit 0.4.0 · scikit-learn 1.9.1 · pandas 3.1.1


## The data

IBM's Telco Customer Churn sample: one row per customer, 19 features, `Churn` = Yes/No. Loaded as pandas reads it,
with no cleaning: cleaning is what we want prefit to tell us about.

In [2]:
import glob

URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
local = glob.glob("/kaggle/input/**/WA_Fn-UseC_-Telco-Customer-Churn.csv", recursive=True)
df = pd.read_csv(local[0] if local else URL)
print(df.shape)
df.dtypes.value_counts()

(7043, 21)


str        18
int64       2
float64     1
Name: count, dtype: int64

In [3]:
blank = df["TotalCharges"].str.strip() == ""
print(f"TotalCharges dtype: {df['TotalCharges'].dtype}; blank values: {blank.sum()}; "
      f"tenure where blank: {sorted(df.loc[blank, 'tenure'].unique())}")

TotalCharges dtype: str; blank values: 11; tenure where blank: [np.int64(0)]


## 1. Findings from human EDAs that change a decision

Listed before running prefit. These are the findings that public EDAs of this dataset on Kaggle commonly report
(paraphrased; the list is ours, check it against the notebooks you trust). Each is kept only if it changes a
decision; "Contract and tenure are the strongest predictors" is useful to know, but by itself changes no step,
so it is listed as context.

| # | finding | the decision it changes |
|---|---|---|
| H1 | `TotalCharges` is text; 11 values are blank | read it as a number, not as 6,500 categories |
| H2 | the 11 blanks are customers with `tenure` = 0 | fill with 0 (no bill yet) instead of imputing a typical value, or drop the rows |
| H3 | `customerID` is a row identifier | drop it |
| H4 | churn is ~26.5% of rows | stratify the split; do not judge the model by accuracy |
| H5 | `TotalCharges` ≈ `tenure` × `MonthlyCharges` | a derived column: drop one of the three for a linear model (collinearity) |
| H6 | "No internet service" repeats `InternetService` = No in six columns | recode to "No" or accept the redundancy (linear models) |
| H7 | month-to-month contracts and short tenure churn most (context) | none by itself |

In [4]:
model = HistGradientBoostingClassifier(random_state=0)
report = pf.analyze(df, "Churn", estimator=model, holdout=0.2)
report

<prefit Report r0: classification (from the target: non-numeric or boolean target) on 'Churn' · 5,634 development rows, 20 features · 2 question(s) open, 0 critical/review decision(s). Next: Answer 2 open question(s) with report.answer({...}); contextual decisions stay out of the model recipe until you do.>

The card shows the three states (needs you / tested / done) and the lines to run next. The first-look table
puts one row per column: what prefit measured, and which decisions mention the column. *signal* is the
threshold-free strength of the feature alone (Gini = 2·AUC − 1, cross-validated on two halves); *leak score* is
the stricter score the leakage screen uses.

In [5]:
report.first_look().sort_values("signal", ascending=False)

,column,kind,missing %,distinct,skew,signal,leak score,decisions
15,Contract,categorical,0.00,3,NaN,0.4743,0.0000,
5,tenure,numeric,0.00,73,0.242943,0.4532,0.2023,d.fill_zero.TotalCharges
12,TechSupport,categorical,0.00,3,NaN,0.4079,0.0000,
9,OnlineSecurity,categorical,0.00,3,NaN,0.3987,0.0000,
8,InternetService,categorical,0.00,3,NaN,0.3931,0.0000,
10,OnlineBackup,categorical,0.00,3,NaN,0.3541,0.0000,
11,DeviceProtection,categorical,0.00,3,NaN,0.3414,0.0000,
18,MonthlyCharges,numeric,0.00,1504,-0.232130,0.3272,0.0151,
17,PaymentMethod,categorical,0.00,4,NaN,0.3176,0.0000,
19,TotalCharges,numeric,0.12,5295,0.962497,0.3068,0.0950,d.fill_zero.TotalCharges


Now the comparison. Each verdict below is computed from the report.

In [6]:
p = report.details.profile
ids = {d.id for d in report.decisions}
tc = pd.to_numeric(df["TotalCharges"], errors="coerce")
rows = []


def add(h, what_prefit_says, verdict):
    rows.append({"finding": h, "prefit": what_prefit_says, "verdict": verdict})


add("H1 TotalCharges is text",
    f"kind = {p.columns['TotalCharges'].kind.value} ({p.columns['TotalCharges'].kind_reason})",
    "caught" if p.columns["TotalCharges"].kind.value == "numeric" else "missed")
sv = p.columns["TotalCharges"].missingness.get("structural", {})
add("H2 blanks are tenure = 0",
    f"missing exactly where {sv.get('column')} = {sv.get('value')} (coverage {sv.get('coverage')}, "
    f"precision {sv.get('precision')}); question q.missing_is_zero.TotalCharges",
    "caught" if "d.fill_zero.TotalCharges" in ids else "missed")
rid = next((f for f in report.details.findings if f.rule_id == "unsupported_columns"), None)
add("H3 customerID is an id",
    f"kind = {p.columns['customerID'].kind.value}; " + (rid.message if rid else "not reported"),
    "caught" if rid and "customerID" in rid.columns and "d.unsupported_columns" not in ids else "caught, as a review item")
mp = report.metric_plan
add("H4 churn 26.5%",
    f"imbalance ratio {p.target.stats['imbalance_ratio']:.2f} (< 3: no imbalance finding); CV stratified; "
    f"metric asked by cost of errors; positive class = '{mp.positive_label}'",
    "caught (through the split and the metric question, not as a finding)")
rho = p.relations.high_corr_pairs
add("H5 TotalCharges ≈ tenure × MonthlyCharges",
    f"pairwise Spearman TotalCharges~tenure {tc.corr(df.tenure, method='spearman'):.2f} < 0.95, no pair flagged; "
    f"the product (corr {tc.corr(df.tenure * df.MonthlyCharges):.4f}) is not tested",
    "missed" if not rho else "caught")
add("H6 'No internet service' repeated",
    "categorical columns are not compared with each other",
    "missed")
add("H7 Contract, tenure strongest (context)",
    "top of the signal ranking: " + ", ".join(report.first_look().sort_values("signal", ascending=False)
                                                .column.head(3)),
    "caught (in the profile, no decision: as it should be)")
comparison = pd.DataFrame(rows)
comparison

,finding,prefit,verdict
0,H1 TotalCharges is text,kind = numeric (numbers stored as text; 7 blank/placeholder value(s) read as missing),caught
1,H2 blanks are tenure = 0,"missing exactly where tenure = 0 (coverage 1.0, precision 1.0); question q.missing_is_zero.TotalCharges",caught
2,H3 customerID is an id,kind = id_like; ['customerID'] identify rows (a distinct value on every row) and are left out of the features; with no repeated values t...,caught
3,H4 churn 26.5%,imbalance ratio 2.77 (< 3: no imbalance finding); CV stratified; metric asked by cost of errors; positive class = 'Yes',"caught (through the split and the metric question, not as a finding)"
4,H5 TotalCharges ≈ tenure × MonthlyCharges,"pairwise Spearman TotalCharges~tenure 0.89 < 0.95, no pair flagged; the product (corr 0.9996) is not tested",missed
5,H6 'No internet service' repeated,categorical columns are not compared with each other,missed
6,"H7 Contract, tenure strongest (context)","top of the signal ranking: Contract, tenure, TechSupport","caught (in the profile, no decision: as it should be)"


**What prefit lists that changes nothing here (noise).** Every item in the roadmap that is not standard
practice, with whether a human EDA asked for it:

In [7]:
pd.DataFrame([{"decision": d.title, "id": d.id, "state": d.state,
               "asked for by H1-H7": {"d.fill_zero.TotalCharges": "H1, H2", "d.metric": "H4"}.get(d.id, "no")}
              for d in report.decisions if not d.standard])

,decision,id,state,asked for by H1-H7
0,Fill missing 'TotalCharges' with 0,d.fill_zero.TotalCharges,needs_you,"H1, H2"
1,Choose the metric from the cost of errors,d.metric,needs_you,H4


**Reading the comparison.** prefit 0.4 catches the findings that change preprocessing (H1, H2, H3, H4) and
lists nothing a human EDA would call useless. (While 0.4 was developed this table had one noise item: the id
column came with a review item "engineer features from it". A string column with a distinct value on every row
is a row identifier, with nothing to derive and no entity to group, so it is now reported as information only;
identifiers with repeated values still raise the item.) It misses the two
findings about *relations between columns* (H5: a column that is a product of two others; H6: categorical
columns that repeat each other). Both matter for linear models and much less for tree ensembles. They are the
next candidates for a rule, and the measured gap is the reason, not a guess.

For comparison, prefit 0.3 on the same frame read `TotalCharges` as a categorical column with 6,531 categories
and no missing values, and proposed target-encoding it; it also asked whether a Yes/No target is a
classification task, and raised a review item for the row id. All three are gone.

## 2. Missing values: TotalCharges

In [8]:
report.show("d.fill_zero.TotalCharges")

**Fill missing 'TotalCharges' with 0  [awaiting your answer · contextual]**

why: 'TotalCharges' is missing in exactly the 7 row(s) where 'tenure' = 0; where 'TotalCharges' is missing, class 'Yes' is 0% of rows; where it is present, 27% (permutation p = 0.164). (Missing exactly where 'tenure' = 0 · TotalCharges = missing_rows_with_value=7 of 7, value_rows_missing=7 of 7)

do:  Answer whether missing 'TotalCharges' means 0.

or:  Median + missing indicator (the default) — right when the value exists but was not recorded.

or:  Drop these rows — right when rows with 'tenure' = 0 will not be scored in production.

or:  Another constant (e.g. the first bill) — right when domain knowledge says what the value is; put it in your own Pipeline.

risk: If the value exists and was simply not recorded, 0 is wrong; median + indicator is safer.

,measure,value,95% interval,method
0,Missing exactly where 'tenure' = 0 · TotalCharges,"missing_rows_with_value=7 of 7, value_rows_missing=7 of 7",,"share of the missing rows of 'TotalCharges' where 'tenure' = 0, and share of the rows with 'tenure' = 0 where 'TotalCharges' is missing ..."
1,Target where missing vs present · TotalCharges,"missing=0, present=0.2657, p_value=0.164",,class share in rows where 'TotalCharges' is missing vs present; permutation test (499 permutations)


**Target where 'TotalCharges' is missing vs present** — The share of class 'Yes' in the 7 rows where 'TotalCharges' is missing and in the other rows (permutation p = 0.164). *Why this figure:* If the target differs where a value is missing, the missingness itself carries signal: keep the indicator, or check that it is not a leak.

In [9]:
pd.Series(p.columns["TotalCharges"].missingness["target"])

kind              class_share
statistic              0.2657
p_value                 0.164
n_permutations            499
class                     Yes
missing                   0.0
observed               0.2657
dtype: object

The 11 blanks sit exactly where `tenure` = 0, and every customer with `tenure` = 0 has a blank: the bill does
not exist yet. None of the 11 churned (27% of the others did), but with 11 rows the permutation test cannot
separate that from chance (p above), so prefit does not raise a separate "informative missingness" review item;
the evidence travels with the question instead.

The question is contextual: the data says *where* the values are missing, not *what* a missing value means. We
answer it: a customer with no tenure has been billed nothing.

In [10]:
report = report.answer(missing_is_zero={"TotalCharges": True}, metric="roc_auc")
print(report.export().describe())

EXPORTED MODEL RECIPE
model: Pipeline(preprocess, estimator)
features: 19 columns
cv: StratifiedKFold(n_splits=5, random_state=0, shuffle=True)
scoring: ['roc_auc', 'average_precision', 'balanced_accuracy']
estimator-dependent default: StandardScaler on numeric features (scale=False to turn off)
included:
  + d.fill_zero.TotalCharges: Fill missing 'TotalCharges' with 0
not included:
  (none)
columns left out (not supported raw in v1; may carry signal):
  · customerID (id_like)
assumptions still open:
  ? positive class = minority class 'Yes'


## 3. The flow

From the raw frame to an honest test score. The report keeps a reference to the frame (checked by fingerprint on
every use), so no step asks for it again:

In [11]:
report = pf.analyze(df, "Churn", estimator=model, holdout=0.2)
report = report.answer(metric="roc_auc", missing_is_zero={"TotalCharges": True}).verify()
fitted = report.export().fit()
review = report.review(fitted)
print(review)

REVIEW — did the roadmap hold?
Task: classification · n=1409

Measured:
- accuracy: 0.7963  [95% CI 0.7743, 0.8169]
- balanced_accuracy: 0.7102  [95% CI 0.6826, 0.7359]
- macro_f1: 0.7221  [95% CI 0.6933, 0.7479]
- minority_recall: 0.5267
- average_precision: 0.6488  [95% CI 0.5995, 0.6993]
- roc_auc: 0.8414  [95% CI 0.8194, 0.8632]
- baseline accuracy (always predict the majority class): 0.7346
- baseline average_precision (random): 0.2654

Checks:
- [HELD] The model beats always predicting the majority class.
    accuracy 79.6% vs baseline 73.5%.
    reference: majority baseline accuracy 73.5%; balanced accuracy of chance = 0.50
- [HELD] The development CV estimate describes the score on the untouched test data.
    Held: CV roc_auc 0.8311 lies inside the test interval [0.8194, 0.8632]; the gap to the test score (0.8414) is within test-set noise.
    reference: 95% bootstrap interval of the test roc_auc; CV: RepeatedStratifiedKFold(n_splits=5, n_repeats=2) — the export splitter, repe

Four lines, and nothing is left open: the review lists no unverified assumption. The Wine flow in the 0.3 guide
took about ten.

`verify()` had nothing conditional to test here (no skewed features past the threshold, no high-cardinality
column once `TotalCharges` is read as a number), so it only records the baseline and the CV estimate that the
review checks against the test interval.

## 4. Does the reported score hold?

The real-data benchmark (`python benchmarks/real_data.py`, results in `benchmarks/results/real_data.md`) runs a
plain flow and the prefit flow on five seeds of this data, each time scoring both models on a final set neither
flow saw. Telco has no duplicates and no leak, so both flows should report honestly and score alike; the
benchmark checks that prefit does not make anything worse where there is nothing to fix.

In [12]:
from pathlib import Path
import json

res = Path("../results/real_data.json")
if res.exists():
    rows = [r for r in json.loads(res.read_text()) if r["dataset"] == "telco"]
    print(pd.DataFrame([{"seed": r["seed"], **{f"{f} {k}": round(r[f][k], 4) for f in ("plain", "prefit")
                                              for k in ("cv", "own_test", "final")}} for r in rows]).to_string())
else:
    print("run python benchmarks/real_data.py telco from the repository root to fill this cell")

   seed  plain cv  plain own_test  plain final  prefit cv  prefit own_test  prefit final
0     0    0.8266          0.8283       0.8469     0.8236           0.8357        0.8423
1     1    0.8252          0.8390       0.8384     0.8308           0.8222        0.8513
2     2    0.8297          0.8207       0.8339     0.8276           0.8309        0.8314
3     3    0.8294          0.8151       0.8403     0.8190           0.8488        0.8379
4     4    0.8277          0.8391       0.8312     0.8278           0.8344        0.8312


## Summary

| question | answer on Telco |
|---|---|
| findings that change a decision, caught | H1 text column, H2 structural blanks, H3 id, H4 imbalance (via split and metric) |
| missed | H5 derived column (product), H6 repeated categorical information |
| noise | none left (the id column's "engineer features" item was removed during 0.4) |
| missing values | `TotalCharges` blank exactly where `tenure` = 0: asked as "does missing mean 0?", with the evidence |
| flow | four lines from raw frame to a reviewed test score |

The two misses are relations between columns, which v1 does not look at. They become rules only if they change a
decision on more than this one dataset.